# 01 · Carga y auditoría

**Objetivo:** unir los 7 CSV y revisar estructura, tipos, faltantes, duplicados, categorías y unidades
**Preguntas que responde:** ¿los datos son consistentes entre años? ¿Qué preguntas de investigación son viables?
**Entrada:** `data/raw/precio_mayorista_fruta-hortaliza_*.csv`
**Salida:** ninguna (solo diagnóstico; la limpieza ocurre en el notebook 02)
**Decisiones tomadas:** se registran en `docs/decisiones.md`

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.carga import listar_archivos, leer_crudo, COLUMNAS_ESPERADAS

RAW = "../data/raw"
FILAS_README = {2020: 192883, 2021: 194776, 2022: 188313, 2023: 188192,
                2024: 189754, 2025: 202029, 2026: 141588}

archivos = listar_archivos(RAW)
print(len(archivos), "archivos encontrados")

dfs = {ruta.name: leer_crudo(ruta) for ruta in archivos}

resumen = pd.DataFrame([
    {
        "archivo": nombre,
        "anio": int(nombre[-8:-4]),
        "filas": len(d),
        "columnas": d.shape[1],
        "encabezado_ok": list(d.columns) == COLUMNAS_ESPERADAS,
    }
    for nombre, d in dfs.items()
])
resumen["filas_readme"] = resumen["anio"].map(FILAS_README)
resumen["coincide_readme"] = resumen["filas"] == resumen["filas_readme"]
print(resumen.to_string(index=False))
print("Total de filas:", resumen["filas"].sum())

# Si algún encabezado no coincide, mostrar la diferencia
for nombre, d in dfs.items():
    if list(d.columns) != COLUMNAS_ESPERADAS:
        print("\nEncabezado distinto en", nombre)
        print("  sobran:", set(d.columns) - set(COLUMNAS_ESPERADAS))
        print("  faltan:", set(COLUMNAS_ESPERADAS) - set(d.columns))

7 archivos encontrados
                                  archivo  anio  filas  columnas  encabezado_ok  filas_readme  coincide_readme
precio_mayorista_fruta-hortaliza_2020.csv  2020 192883        14           True        192883             True
precio_mayorista_fruta-hortaliza_2021.csv  2021 194776        14           True        194776             True
precio_mayorista_fruta-hortaliza_2022.csv  2022 188313        14           True        188313             True
precio_mayorista_fruta-hortaliza_2023.csv  2023 188192        14           True        188192             True
precio_mayorista_fruta-hortaliza_2024.csv  2024 189754        14           True        189754             True
precio_mayorista_fruta-hortaliza_2025.csv  2025 202029        14           True        202029             True
precio_mayorista_fruta-hortaliza_2026.csv  2026 141588        14           True        141588             True
Total de filas: 1297535


In [2]:
df = pd.concat(
    [d.assign(archivo=nombre) for nombre, d in dfs.items()],
    ignore_index=True,
)
del dfs  # libera memoria

print(df.shape)
print(df.head(3).T)   # transpuesta para leer cómodo las 14 columnas

(1297535, 15)
                                                                    0  \
Fecha                                                      2020-01-02   
ID region                                                           5   
Region                                           Región de Valparaíso   
Mercado                                          Femacal de La Calera   
Subsector                                     Hortalizas y tubérculos   
Producto                                                       Acelga   
Variedad / Tipo                                       Sin especificar   
Calidad                                                       Primera   
Unidad de comercializacion               $/docena de atados (6 kilos)   
Origen                                          Provincia de Quillota   
Volumen                                                           120   
Precio minimo                                               1800,0000   
Precio maximo                        

In [3]:
# --- Fechas ---
fecha = pd.to_datetime(df["Fecha"], format="%Y-%m-%d", errors="coerce")
print("Fechas no interpretables:", fecha.isna().sum())

df["anio_archivo"] = df["archivo"].str[-8:-4].astype(int)
df["anio_fecha"] = fecha.dt.year

print("\n¿Cada archivo contiene solo fechas de su año?")
print(pd.crosstab(df["anio_archivo"], df["anio_fecha"]))

print("\nRango de fechas por archivo:")
print(fecha.groupby(df["anio_archivo"]).agg(["min", "max"]))

# --- Números ---
columnas_num = ["Volumen", "Precio minimo", "Precio maximo", "Precio promedio"]
print("\nColumnas numéricas:")
for c in columnas_num:
    limpio = df[c].str.replace(",", ".", regex=False).str.strip()
    num = pd.to_numeric(limpio, errors="coerce")
    vacios = (limpio == "").sum()
    no_numericos = num.isna().sum() - vacios
    print(f"{c:15s} vacíos={vacios:<7} no_numéricos={no_numericos:<7} "
          f"min={num.min():<12} max={num.max()}")

Fechas no interpretables: 0

¿Cada archivo contiene solo fechas de su año?
anio_fecha      2020    2021    2022    2023    2024    2025    2026
anio_archivo                                                        
2020          192883       0       0       0       0       0       0
2021               0  194776       0       0       0       0       0
2022               0       0  188313       0       0       0       0
2023               0       0       0  188192       0       0       0
2024               0       0       0       0  189754       0       0
2025               0       0       0       0       0  202029       0
2026               0       0       0       0       0       0  141588

Rango de fechas por archivo:
                    min        max
anio_archivo                      
2020         2020-01-02 2020-12-31
2021         2021-01-04 2021-12-31
2022         2022-01-03 2022-12-30
2023         2023-01-03 2023-12-29
2024         2024-01-02 2024-12-31
2025         2025-01-01 2025-

In [4]:
print(f"{df.memory_usage(deep=True).sum() / 1024**2:.0f} MB en memoria")
print(df["ID region"].nunique(), "valores distintos de ID region")

1247 MB en memoria
9 valores distintos de ID region
